# Amazon Product Review Rating Prediction

Self-contained Colab notebook: không cần clone repo, không cần file script ngoài.

Bài toán: dự đoán rating 1-5 sao từ review sản phẩm Amazon.

Model: `Word Embedding -> TextCNN -> BiLSTM -> Attention -> MLP`.

## 1. Setup GPU and dependencies

In [ ]:
!nvidia-smi
!pip -q install datasets scikit-learn tqdm tensorboard seaborn

import json
import math
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configuration

`quick_mode=True` để chạy thử nhanh. Khi pipeline ổn, đổi thành `False` để train nghiêm túc hơn.

In [ ]:
@dataclass
class TrainConfig:
    dataset_name: str = 'SetFit/amazon_reviews_multi_en'
    text_column: str = 'text'
    label_column: str = 'label'
    train_split: str = 'train'
    validation_split: str = 'test'
    output_dir: str = 'artifacts/amazon_reviews_hybrid_colab'

    num_classes: int = 5
    max_vocab_size: int = 30000
    min_freq: int = 2
    max_length: int = 128

    embedding_dim: int = 128
    hidden_dim: int = 192
    cnn_num_filters: int = 96
    cnn_kernel_sizes: str = '3,5,7'
    lstm_hidden_dim: int = 96
    lstm_layers: int = 1
    dropout: float = 0.4

    batch_size: int = 128
    epochs: int = 3
    learning_rate: float = 7e-4
    weight_decay: float = 1e-3
    early_stopping_patience: int = 2
    clip_grad_norm: float = 1.0

    quick_mode: bool = True
    quick_train_size: int = 30000
    quick_valid_size: int = 5000
    seed: int = 42


config = TrainConfig()

# Full experiment suggestion. Uncomment these lines for a stronger run.
# config.quick_mode = False
# config.max_vocab_size = 50000
# config.max_length = 256
# config.embedding_dim = 200
# config.hidden_dim = 256
# config.cnn_num_filters = 128
# config.lstm_hidden_dim = 128
# config.lstm_layers = 2
# config.batch_size = 64
# config.epochs = 10
# config.learning_rate = 5e-4
# config.dropout = 0.5
# config.early_stopping_patience = 3

Path(config.output_dir).mkdir(parents=True, exist_ok=True)
config

## 3. Utilities: seed, tokenizer, vocabulary, encoder

In [ ]:
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
PAD_ID = 0
UNK_ID = 1


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def simple_tokenize(text: str) -> List[str]:
    text = text.lower().strip()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r"[^a-z0-9'\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text)
    if not text:
        return []
    return text.split()


def build_vocab(texts: List[str], max_vocab_size: int, min_freq: int) -> Dict[str, int]:
    counter = Counter()
    for text in tqdm(texts, desc='Building vocab'):
        counter.update(simple_tokenize(text))

    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    remaining_slots = max_vocab_size - len(vocab)
    for word, freq in counter.most_common(remaining_slots):
        if freq < min_freq:
            continue
        vocab[word] = len(vocab)
    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_length: int) -> Tuple[List[int], List[int]]:
    tokens = simple_tokenize(text)
    input_ids = [vocab.get(token, UNK_ID) for token in tokens]
    input_ids = input_ids[:max_length]
    attention_mask = [1] * len(input_ids)

    padding_length = max_length - len(input_ids)
    if padding_length > 0:
        input_ids += [PAD_ID] * padding_length
        attention_mask += [0] * padding_length
    return input_ids, attention_mask


def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


set_seed(config.seed)
device = get_device()
device

## 4. Load Amazon Reviews dataset

In [ ]:
raw_dataset = load_dataset(config.dataset_name)
raw_dataset

In [ ]:
train_split = raw_dataset[config.train_split]
valid_split = raw_dataset[config.validation_split]

if config.quick_mode:
    train_split = train_split.shuffle(seed=config.seed).select(range(config.quick_train_size))
    valid_split = valid_split.shuffle(seed=config.seed).select(range(config.quick_valid_size))

train_texts = list(train_split[config.text_column])
train_labels = list(train_split[config.label_column])
valid_texts = list(valid_split[config.text_column])
valid_labels = list(valid_split[config.label_column])

print('Train samples:', len(train_texts))
print('Validation samples:', len(valid_texts))
print('Example text:', train_texts[0][:300])
print('Example label:', train_labels[0])

## 5. Build vocabulary and dataset

In [ ]:
vocab = build_vocab(
    train_texts,
    max_vocab_size=config.max_vocab_size,
    min_freq=config.min_freq,
)
print('Vocabulary size:', len(vocab))
list(vocab.items())[:20]

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], vocab: Dict[str, int], max_length: int):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_length = max_length

        if len(texts) != len(labels):
            raise ValueError('texts and labels must have the same length')

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int):
        input_ids, attention_mask = encode_text(self.texts[idx], self.vocab, self.max_length)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = ReviewDataset(train_texts, train_labels, vocab, config.max_length)
valid_dataset = ReviewDataset(valid_texts, valid_labels, vocab, config.max_length)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

batch = next(iter(train_loader))
{key: value.shape for key, value in batch.items()}

## 6. Define Hybrid CNN-BiLSTM-Attention model

In [ ]:
class HybridCnnLstmAttentionClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int,
        hidden_dim: int,
        cnn_num_filters: int,
        cnn_kernel_sizes: str,
        lstm_hidden_dim: int,
        lstm_layers: int,
        dropout: float,
        padding_idx: int = 0,
    ):
        super().__init__()
        kernel_sizes = [int(size.strip()) for size in cnn_kernel_sizes.split(',')]

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=cnn_num_filters,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
            )
            for kernel_size in kernel_sizes
        ])

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

        feature_dim = embedding_dim + (cnn_num_filters * len(kernel_sizes)) + (lstm_hidden_dim * 2)

        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        mask = attention_mask.unsqueeze(-1).float()
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)
        masked_embedded = embedded * mask

        mean_feature = masked_embedded.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        cnn_input = masked_embedded.transpose(1, 2)
        conv_mask = attention_mask.unsqueeze(1).bool()
        cnn_features = []
        for conv in self.convs:
            conv_output = F.gelu(conv(cnn_input))
            if conv_output.size(-1) > attention_mask.size(1):
                conv_output = conv_output[:, :, :attention_mask.size(1)]
            elif conv_output.size(-1) < attention_mask.size(1):
                pad_size = attention_mask.size(1) - conv_output.size(-1)
                conv_output = F.pad(conv_output, (0, pad_size))
            conv_output = conv_output.masked_fill(~conv_mask, -1e4)
            cnn_features.append(conv_output.max(dim=2).values)
        cnn_feature = torch.cat(cnn_features, dim=1)

        lstm_output, _ = self.lstm(masked_embedded)
        attention_scores = self.attention(lstm_output).squeeze(-1)
        attention_scores = attention_scores.masked_fill(attention_mask == 0, -1e4)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)
        attention_feature = (lstm_output * attention_weights).sum(dim=1)

        features = torch.cat([mean_feature, cnn_feature, attention_feature], dim=1)
        return self.classifier(features)

    def get_word_embedding_matrix(self):
        return self.embedding.weight.detach()

    def get_sentence_embedding(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        mask = attention_mask.unsqueeze(-1).float()
        embedded = self.embedding(input_ids)
        masked_embedded = embedded * mask
        return masked_embedded.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)


model = HybridCnnLstmAttentionClassifier(
    vocab_size=len(vocab),
    num_classes=config.num_classes,
    embedding_dim=config.embedding_dim,
    hidden_dim=config.hidden_dim,
    cnn_num_filters=config.cnn_num_filters,
    cnn_kernel_sizes=config.cnn_kernel_sizes,
    lstm_hidden_dim=config.lstm_hidden_dim,
    lstm_layers=config.lstm_layers,
    dropout=config.dropout,
    padding_idx=PAD_ID,
).to(device)

trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'Trainable parameters: {trainable_params:,}')

with torch.no_grad():
    test_logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
print('Logit shape:', test_logits.shape)

## 7. Training and evaluation functions

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=1)
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average='macro', zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average='weighted', zero_division=0
    )

    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_precision,
        'weighted_recall': weighted_recall,
        'weighted_f1': weighted_f1,
        'labels': all_labels,
        'predictions': all_predictions,
    }


def save_artifacts(model, vocab, config, output_dir: Path, best_macro_f1: float):
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), output_dir / 'model.pt')
    with open(output_dir / 'vocab.json', 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)
    saved_config = asdict(config)
    saved_config['vocab_size'] = len(vocab)
    saved_config['best_macro_f1'] = best_macro_f1
    with open(output_dir / 'config.json', 'w', encoding='utf-8') as f:
        json.dump(saved_config, f, ensure_ascii=False, indent=2)


def save_history(history, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    with open(output_dir / 'training_history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)


## 8. Train model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=1,
)

output_dir = Path(config.output_dir)
best_macro_f1 = 0.0
epochs_without_improvement = 0
history = []

for epoch in range(1, config.epochs + 1):
    model.train()
    total_train_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch}/{config.epochs}')
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.clip_grad_norm)
        optimizer.step()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=f'{loss.item():.4f}')

    train_loss = total_train_loss / len(train_loader)
    valid_metrics = evaluate(model, valid_loader, criterion, device)
    scheduler.step(valid_metrics['loss'])

    epoch_history = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': valid_metrics['loss'],
        'val_accuracy': valid_metrics['accuracy'],
        'val_macro_precision': valid_metrics['macro_precision'],
        'val_macro_recall': valid_metrics['macro_recall'],
        'val_macro_f1': valid_metrics['macro_f1'],
        'val_weighted_precision': valid_metrics['weighted_precision'],
        'val_weighted_recall': valid_metrics['weighted_recall'],
        'val_weighted_f1': valid_metrics['weighted_f1'],
        'learning_rate': optimizer.param_groups[0]['lr'],
    }
    history.append(epoch_history)
    save_history(history, output_dir)

    print(
        f"Epoch {epoch}/{config.epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={valid_metrics['loss']:.4f} | "
        f"acc={valid_metrics['accuracy']:.4f} | "
        f"macro_f1={valid_metrics['macro_f1']:.4f} | "
        f"weighted_f1={valid_metrics['weighted_f1']:.4f}"
    )

    if valid_metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = valid_metrics['macro_f1']
        epochs_without_improvement = 0
        save_artifacts(model, vocab, config, output_dir, best_macro_f1)
        print(f'New best model saved. macro_f1={best_macro_f1:.4f}')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= config.early_stopping_patience:
            print('Early stopping triggered.')
            break

print('Training completed. Best macro F1:', best_macro_f1)

## 9. Analyze training history

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Train loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='Validation loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['val_macro_f1'], marker='o', label='Macro F1')
axes[1].plot(history_df['epoch'], history_df['val_weighted_f1'], marker='o', label='Weighted F1')
axes[1].plot(history_df['epoch'], history_df['val_accuracy'], marker='o', label='Accuracy')
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['learning_rate'], marker='o', color='tab:purple')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning rate')

plt.tight_layout()
plt.show()

## 10. Post-training EDA

Các biểu đồ này giúp hiểu dữ liệu và giải thích kết quả model sau khi train.

In [ ]:
rating_names = {0: '1-star', 1: '2-star', 2: '3-star', 3: '4-star', 4: '5-star'}

eda_df = pd.DataFrame({
    'text': train_texts,
    'label': train_labels,
})
eda_df['rating'] = eda_df['label'].map(rating_names)
eda_df['char_len'] = eda_df['text'].str.len()
eda_df['token_len'] = eda_df['text'].apply(lambda text: len(simple_tokenize(text)))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=eda_df, x='rating', order=list(rating_names.values()), ax=axes[0], palette='viridis')
axes[0].set_title('Training Label Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Number of reviews')

sns.boxplot(data=eda_df, x='rating', y='token_len', order=list(rating_names.values()), ax=axes[1], palette='viridis')
axes[1].set_title('Review Length by Rating')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Token length')
axes[1].set_ylim(0, eda_df['token_len'].quantile(0.95))

plt.tight_layout()
plt.show()

display(eda_df.groupby('rating')[['token_len', 'char_len']].describe().round(2))

In [ ]:
top_words = Counter()
for text in tqdm(train_texts, desc='Counting top words'):
    top_words.update(simple_tokenize(text))

top_word_df = pd.DataFrame(top_words.most_common(30), columns=['word', 'count'])

plt.figure(figsize=(12, 8))
sns.barplot(data=top_word_df, y='word', x='count', palette='mako')
plt.title('Top 30 Most Frequent Tokens')
plt.xlabel('Count')
plt.ylabel('Token')
plt.tight_layout()
plt.show()

display(top_word_df.head(15))

## 11. Classification report and confusion matrix

In [ ]:
best_state = torch.load(output_dir / 'model.pt', map_location=device)
model.load_state_dict(best_state)
final_metrics = evaluate(model, valid_loader, criterion, device)

target_names = ['1-star', '2-star', '3-star', '4-star', '5-star']
print(classification_report(final_metrics['labels'], final_metrics['predictions'], target_names=target_names, zero_division=0))

cm = confusion_matrix(final_metrics['labels'], final_metrics['predictions'])
cm_normalized = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens', xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

## 12. Error analysis and prediction confidence

In [ ]:
model.eval()
analysis_rows = []

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(valid_loader, desc='Collecting validation predictions')):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        confidence, preds = torch.max(probs, dim=1)

        start_idx = batch_idx * config.batch_size
        for i in range(labels.size(0)):
            row_idx = start_idx + i
            analysis_rows.append({
                'text': valid_texts[row_idx],
                'true_label': int(labels[i].cpu()),
                'pred_label': int(preds[i].cpu()),
                'confidence': float(confidence[i].cpu()),
                'absolute_error': abs(int(labels[i].cpu()) - int(preds[i].cpu())),
            })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df['true_rating'] = analysis_df['true_label'].map(rating_names)
analysis_df['pred_rating'] = analysis_df['pred_label'].map(rating_names)
analysis_df['correct'] = analysis_df['true_label'] == analysis_df['pred_label']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(data=analysis_df, x='confidence', hue='correct', bins=30, ax=axes[0], kde=True)
axes[0].set_title('Prediction Confidence Distribution')
axes[0].set_xlabel('Max softmax probability')

off_by_one_accuracy = (analysis_df['absolute_error'] <= 1).mean()
error_df = analysis_df['absolute_error'].value_counts().sort_index().reset_index()
error_df.columns = ['absolute_error', 'count']
sns.barplot(data=error_df, x='absolute_error', y='count', ax=axes[1], palette='crest')
axes[1].set_title(f'Rating Distance Error | Off-by-one acc = {off_by_one_accuracy:.3f}')
axes[1].set_xlabel('|true rating - predicted rating|')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

display(analysis_df.sort_values('confidence', ascending=False).head(10)[['true_rating', 'pred_rating', 'confidence', 'text']])
display(analysis_df[~analysis_df['correct']].sort_values('confidence', ascending=False).head(10)[['true_rating', 'pred_rating', 'confidence', 'absolute_error', 'text']])

## 13. Inference examples

In [ ]:
def predict_rating(text: str):
    model.eval()
    input_ids, attention_mask = encode_text(text, vocab, config.max_length)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probabilities = torch.softmax(logits, dim=1).squeeze(0)
        pred = torch.argmax(probabilities).item()
    return pred + 1, probabilities.cpu().numpy()


examples = [
    'This product is amazing. The quality is excellent and I would definitely buy it again.',
    'It arrived broken and the support team was not helpful at all.',
    'The item is okay for the price, but the build quality could be better.',
    'The product works, but the battery life is shorter than advertised and the packaging felt cheap.',
    'I bought this as a gift and it exceeded expectations. Fast delivery, premium feel, and great value.',
    'Completely useless. It stopped working after two days and the return process was frustrating.',
    'Decent for casual use, but I would not recommend it for anyone who needs something durable.',
]

for text in examples:
    rating, probs = predict_rating(text)
    print('\nText:', text)
    print('Predicted rating:', rating)
    print('Probabilities:', np.round(probs, 3))

## 14. Export word embeddings to TensorBoard Projector

Ở đây lấy `model.embedding.weight`: mỗi hàng là vector của một word trong vocabulary.

In [ ]:
log_dir = Path('runs/amazon_review_word_embeddings')
log_dir.mkdir(parents=True, exist_ok=True)

embedding_matrix = model.get_word_embedding_matrix().detach().cpu()
id_to_word = {idx: word for word, idx in vocab.items()}

top_k = min(10000, embedding_matrix.shape[0])
selected_ids = list(range(top_k))
selected_embeddings = embedding_matrix[selected_ids]
metadata = [id_to_word[idx] for idx in selected_ids]

writer = SummaryWriter(str(log_dir))
writer.add_embedding(
    mat=selected_embeddings,
    metadata=metadata,
    tag='amazon_review_word_embeddings',
)
writer.close()

print(f'Exported {top_k} word embeddings to {log_dir}')

## 15. Launch TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 16. Download artifacts

In [ ]:
!zip -r amazon_reviews_hybrid_colab_artifacts.zip artifacts runs

from google.colab import files
files.download('amazon_reviews_hybrid_colab_artifacts.zip')